# Generating type 4 clones with LLMs

### Configuration

In [ ]:
import src.clone_gen as cg 
DeepSeek = "deepseek-r1:14b" 
Gemma3 = "gemma3:latest" 
Gpt20b= "gpt-oss:20b"
LLama3 = "llama3.1:latest" 
ALL_MODELS = [DeepSeek, Gemma3, LLama3, Gpt20b] 


DATASET_PATH = "../dataset/dataset.json"
OUT_PATH      = "../results/bigcodebench_llm_clones.json"
OLLAMA_MODEL =  DeepSeek
NL_MODEL = LLama3    
CODE_MODEL = LLama3    

# Generation settings
LLM_OPTS = {
    "temperature": 0.1,      # lower = more deterministic, higher = more creative
    "top_p": 0.95,            # nucleus sampling: consider only tokens with cumulative prob ≤ 0.95
    "repeat_penalty": 1.1,   # penalizes repeated tokens to avoid repetition
    "num_predict": 1500,       # max number of tokens the model will generate
}
 
from src.clone_gen import REMOTE_OLLAMA, N_ENTRIES, CLONES_PER_ENTRY 
cg.REMOTE_OLLAMA = REMOTE_OLLAMA

### Test connection to server

In [2]:
from src.clone_gen import call_ollama_chat
messages = [
    {"role": "user", "content": "Are you up and running, answer in one word."}
] 
response = call_ollama_chat(messages, OLLAMA_MODEL, LLM_OPTS)
print(response)

Yes


## Generation of clones

### Generation of Additional Fields

In [3]:
# from src.clone_gen import add_generated_fields

# add_generated_fields(
#     dataset_path=DATASET_PATH, 
#     n_entries=N_ENTRIES
# )

### Translating Source code 

In [4]:
# from src.clone_gen import add_generated_translation

# add_generated_translation(
#     dataset_path=DATASET_PATH,
#     code_model=CODE_MODEL,
#     llm_opts=LLM_OPTS,
#     n_entries=N_ENTRIES,
#     language="Java"
# )

## Generating clones

In [ ]:
import random
from itertools import combinations
from src.clone_gen import run_clone_generation
from src.utils import select_balanced_combinations

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

CONTEXTS = ["ast", "code", "test", "complete"]
REFACS = [f"refac_{i}" for i in range(1, 8)]  # refac_1..refac_7
STRATEGIES = ["zero-shot", "cot"] 
#all_models = ALL_MODELS
all_models = [DeepSeek]
# --- Generate only combinations of size 3 ---
all_combinations = list(combinations(REFACS, 3)) 

# --- Choose balanced subset ---
NUM_COMBINATIONS_TO_USE = 7  # can be any number up to len(all_combinations)
subset_combinations = select_balanced_combinations(all_combinations, NUM_COMBINATIONS_TO_USE, REFACS, RANDOM_SEED)

# --- Iterate over combinations for all strategy+context pairs ---
for model in all_models:
    for strategy in STRATEGIES:
        for context in CONTEXTS:
            for run_idx, refac_tuple in enumerate(subset_combinations, 1):
                selected_refacs = list(refac_tuple)
                print(f"\n=== Generating with {model}, strategy={strategy}, context={context} and {selected_refacs}")

                run_clone_generation(
                    dataset_path=DATASET_PATH,
                    out_path=OUT_PATH,
                    n_entries=N_ENTRIES, 
                    clones_per_entry=CLONES_PER_ENTRY,
                    ollama_model=OLLAMA_MODEL,
                    llm_opts=LLM_OPTS,
                    context=context,
                    refacs=selected_refacs,
                    strategy=strategy,
                )



Balanced refactor usage:
  refac_1: 3
  refac_2: 3
  refac_3: 3
  refac_4: 3
  refac_5: 3
  refac_6: 3
  refac_7: 3


=== Generating for strategy=zero-shot, context=ast and ['refac_1', 'refac_4', 'refac_5']

Generating clones 1/12 for BigCodeBench/58

Generating clones 2/12 for BigCodeBench/151

Generating clones 3/12 for BigCodeBench/156

Generating clones 4/12 for BigCodeBench/168

Generating clones 5/12 for BigCodeBench/179

Generating clones 6/12 for BigCodeBench/213

Generating clones 7/12 for BigCodeBench/253

Generating clones 8/12 for BigCodeBench/261

Generating clones 9/12 for BigCodeBench/271

Generating clones 10/12 for BigCodeBench/375

Generating clones 11/12 for BigCodeBench/411

Generating clones 12/12 for BigCodeBench/437

=== Generating for strategy=zero-shot, context=ast and ['refac_2', 'refac_6', 'refac_7']

Generating clones 1/12 for BigCodeBench/58

Generating clones 2/12 for BigCodeBench/151

Generating clones 3/12 for BigCodeBench/156

Generating clones 4/12 fo